# pymor.solvers

In [ ]:
#| default_exp pymor.solvers

## Imports

In [ ]:
#| export
from concurrent.futures import ThreadPoolExecutor
from functools import partial
from time import time as clock
import logging

import numpy as np
from fastcore.basics import ifnone
from scipy.sparse.linalg import LinearOperator, gcrotmk, spilu
from pymor.solvers.interface import Solver
from pymor.core.exceptions import InversionError

from pylgs.pymor.operators import to_matrix

In [ ]:
#|export
logger = logging.getLogger(__name__)

## Testing

## API

### BlockSpILUPreconditioner -

In [ ]:
#|export
def diagonal_blocks(mat, block_size):
    n = mat.shape[0]
    blocks = []
    for i in range(0, n, block_size):
        blocks.append(mat[i:i+block_size, i:i+block_size])
    return blocks

In [ ]:
#|export
class BlockSpILUPreconditioner(Solver):
    def __init__(self, matrix=None, block_size=None, spilu_drop_tol=1e-4, spilu_fill_factor=20, spilu_drop_rule=None, spilu_permc_spec='COLAMD', n_workers=None, single_precision=False):
        n_workers = ifnone(n_workers, 4)
        self._ilu = None
        self.spilu = partial(spilu, drop_tol=spilu_drop_tol, fill_factor=spilu_fill_factor, drop_rule=spilu_drop_rule, permc_spec=spilu_permc_spec)
        self.__auto_init(locals())

    def factorize(self, mu):
        if self.matrix is None or self.block_size is None:
            raise ValueError("Matrix and block size must be specified for the preconditioner.")
        matrix = to_matrix(self.matrix.assemble(mu)).tocsc()
        if self.single_precision:
            single_type = np.complex64 if np.iscomplexobj(matrix) else np.float32
            matrix = matrix.astype(single_type, casting='unsafe')
        blocks = diagonal_blocks(matrix, self.block_size)
        self._n_blocks = len(blocks)
        self._blocks = blocks
        if self._ilu is None: self._ilu = [None] * len(blocks)
        new_blocks = [i for i in range(len(blocks)) if self._ilu[i] is None]
        tick = clock()
        if self.n_workers == 1:
            new_ilus = [self.spilu(blocks[i]) for i in new_blocks]
        else:
            with ThreadPoolExecutor(max_workers=self.n_workers) as ex:
                new_ilus = list(ex.map(self.spilu, [blocks[i] for i in new_blocks]))
        logger.info(f'Factoring {len(new_blocks)} new blocks out of {len(blocks)} took {clock()-tick:.2f} seconds')
        for idx, ilu in zip(new_blocks, new_ilus):
            self._ilu[idx] = ilu

    def operator(self, mu):
        self.factorize(mu)
        self._n_solves = 0

        def ilu_block_solve(v):
            v = v.reshape(self._n_blocks, self.block_size)
            if self.single_precision:
                single_type = np.complex64 if np.iscomplexobj(v) else np.float32
                v = v.astype(single_type, casting='unsafe')
            if self.n_workers == 1:
                result = np.array([self._ilu[i].solve(v[i]) for i in range(self._n_blocks)])
            else:
                with ThreadPoolExecutor(max_workers=self.n_workers) as ex:
                    result = np.array(list(ex.map(lambda i: self._ilu[i].solve(v[i]), range(self._n_blocks))))
            self._n_solves += 1
            return result
        
        logger.info(f'{self._n_blocks} preconditioner blocks of size {self.block_size}, type {self._ilu[0].L.dtype}, and density {self._blocks[0].nnz / (self.block_size**2):.2e}')
        return LinearOperator((self._n_blocks*self.block_size, self._n_blocks*self.block_size), ilu_block_solve)

In [ ]:
# from sksparse.umfpack import umf_factor, UMFFactor

In [ ]:
# #|export
# from fastcore.parallel import parallel
# from functools import partial
# from sksparse.umfpack import umf_factor, UMFFactor

# class BlockUMFPreconditioner(Solver):
#     def __init__(self, matrix=None, block_size=None):
#         self.matrix = matrix
#         self.block_size = block_size
#         self._ilu = None
#         self._umf = None

#     def factorize(self, mu):
#         if self.matrix is None or self.block_size is None:
#             raise ValueError("Matrix and block size must be specified for the preconditioner.")
#         matrix = to_matrix(self.matrix.assemble(mu)).tocsc()
#         blocks = diagonal_blocks(matrix, self.block_size)
#         self._n_blocks = len(blocks)
#         self._blocks = blocks
#         if self._umf is None: self._umf = UMFFactor(blocks[0])
#         if self._ilu is None: self._ilu = [None] * len(blocks)
#         new_blocks = [i for i in range(len(blocks)) if self._ilu[i] is None]
#         tick = clock()
#         # new_ilus = parallel(
#         #     lambda A: self._umf.copy().factorize(A=A),
#         #     [blocks[i] for i in new_blocks],
#         #     threadpool=True,
#         #     n_workers=4,
#         # )
#         umf = self._umf.copy()
#         umf.factorize(blocks[new_blocks[0]])
#         umf = self._umf.copy()
#         umf.factorize(blocks[new_blocks[1]])
#         new_ilus = [self._umf.copy().factorize(A=blocks[i]) for i in new_blocks]
#         logger.info(f'BlockUMFPreconditioner: Factoring {len(new_blocks)} new blocks out of {len(blocks)} took {clock()-tick:.2f} seconds')
#         for idx, ilu in zip(new_blocks, new_ilus):
#             self._ilu[idx] = ilu

#     def operator(self, mu):
#         self.factorize(mu)
#         ilu = self._ilu
#         self._n_solves = 0

#         def ilu_block_solve(v):
#             v = v.reshape(self._n_blocks, self.block_size)
#             result = np.empty_like(v).astype(self._blocks[0].dtype)
#             for i in range(self._n_blocks):
#                 result[i] = ilu[i].solve(v[i])
#             self._n_solves += 1
#             return result.reshape(-1)
        
#         logger.info(f'BlockUMFPreconditioner: {self._n_blocks} preconditioner blocks of size {self.block_size}, type {self._ilu[0].L.dtype}, and density {self._blocks[0].nnz / (self.block_size**2):.2e}')
#         return LinearOperator((self._n_blocks*self.block_size, self._n_blocks*self.block_size), ilu_block_solve)

### ScipyGCROTMKSolver -

In [ ]:
#| export
class ScipyGCROTMKSolver(Solver):

    def __init__(self, check_finite=None, atol=0., rtol=1e-5, maxiter=1000, gcrotmk_m=20, preconditioner=None):
        super().__init__()
        if isinstance(preconditioner, (list, tuple)):
            preconditioner = preconditioner[0](**preconditioner[1:])
        self.__auto_init(locals())

    def _solve(self, operator, V, mu, initial_guess):
        tick = clock()
        operator = operator.assemble(mu)
        tock = clock()
        logger.info('Operator assembly took {:.2f} seconds'.format(tock - tick))
        matrix = to_matrix(operator).tocsc()
        logger.info('Operator conversion to CSC took {:.2f} seconds'.format(clock() - tock))
        V = V.to_numpy()
        initial_guess = initial_guess.to_numpy() if initial_guess is not None else None
        promoted_type = np.promote_types(matrix.dtype, V.dtype)
        logger.info(f'Operator setup completed in {clock() - tick:.2f} seconds')
        logger.info(f'Solving {V.shape[1]} linear systems of size {matrix.shape[0]} using gcrotmk (m={self.gcrotmk_m}, rtol={self.rtol}, atol={self.atol})')

        tic = clock()
        precond_op = self.preconditioner.operator(mu) if self.preconditioner is not None else None
        logger.info(f'Preconditioner setup completed in {clock() - tic:.2f} seconds')
        n_iterations = 0
        R = np.empty((matrix.shape[1], V.shape[1]), dtype=promoted_type, order='F')

        def callback(xk):
            nonlocal n_iterations
            n_iterations += 1

        for i in range(V.shape[1]):
            tic = clock()
            n_iterations = 0
            R[:, i], info = gcrotmk(matrix, V[:, i], initial_guess[:, i] if initial_guess is not None else None,
                                     atol=self.atol, rtol=self.rtol, maxiter=self.maxiter, M=precond_op, m=self.gcrotmk_m, callback=callback)
            if info != 0:
                if info > 0:
                    raise InversionError(f'gcrotmk failed to converge after {info} iterations')
                else:
                    raise InversionError(f'gcrotmk failed with error code {info} (illegal input or breakdown)')

            logger.info(f'gcrotmk converged in {n_iterations} iterations for vector {i+1}/{V.shape[1]}, with {self.preconditioner._n_solves if self.preconditioner is not None else 0} preconditioner solves in {clock() - tic:.2f} seconds')

        if self.check_finite:
            if not np.isfinite(np.sum(R)):
                raise InversionError('Result contains non-finite values')

        return operator.source.from_numpy(R), {}